In [ ]:
import sys
print(sys.version)

# AdvTG — end-to-end pipeline

Adversarial HTTP traffic generation vs. DL malicious-traffic detectors, run stage by stage:

1. **Dataset** → 2. **Detectors** (token + image) → 3. **LLM finetune** (Unsloth) → 4. **PPO** adversarial generation

**Where this runs:** the kernel is a **Colab GPU VM** — whether you open this on colab.research.google.com or connect VS Code to a Colab runtime, the cells execute on Colab's VM. So:

- **Requires Python 3.11** (the top cell prints the version). On Colab's default 3.13 the pinned 2024 stack won't build; 3.11 works.
- The VM only has what's in the **git remote**. **Push your branch first**; the Setup cell clones it (and `git pull`s on re-runs). Local VS Code edits do **not** sync to the VM.
- Give the runtime a GPU: Colab ▸ Runtime ▸ Change runtime type ▸ **T4**.

**One install, one kernel, no restarts.** The **Install** cell below pins the whole 2024-era stack that *all four stages share*, so you can run the notebook top to bottom in a single kernel — just execute the cells in order.

In [ ]:
import os, sys, subprocess

# Setup as a function. Light + idempotent (git pull + chdir + sys.path — NO torch import),
# so it's safe to call at the top of any stage via prepare(). Run this cell once to define it.
GIT_URL = "https://github.com/TejaswiMN/AdvTG.git"
BRANCH  = "punyam"

def setup():
    """Clone/pull the repo, cd into it, make it importable. Sets global REPO."""
    global REPO
    if os.path.isfile(os.path.join(os.getcwd(), "gen_synthetic_data.py")):
        REPO = os.getcwd()                                   # already inside the repo
    else:
        # Kaggle's writable dir is /kaggle/working; Colab's is /content.
        REPO = "/kaggle/working/AdvTG" if os.path.isdir("/kaggle") else "/content/AdvTG"
        if os.path.isdir(os.path.join(REPO, ".git")):
            subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
        else:
            subprocess.run(["git", "clone", "-b", BRANCH, GIT_URL, REPO], check=True)
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)                             # make the DL package importable
    return REPO

setup()
print("repo:", REPO, "| has gen_synthetic_data.py:", os.path.isfile("gen_synthetic_data.py"))

## Config — the settings live in `config()`

To change a setting (dataset source, sizes, epochs…), edit the value **inside the `config()` function** below and re-run this cell. Everything downstream reads `dataset/train_data2.json`.

`setup()` and `config()` are light, torch-free helpers; `prepare()` runs both and is called at the top of every stage — so each stage is self-contained and you never have to scroll back up to re-run Setup/Config.

In [ ]:
import os

# Config as a function (light + idempotent, no torch import). prepare() = setup() + config(),
# called at the top of every stage so each stage is self-contained. To change a setting,
# edit the value inside config() and re-run this cell.
def config():
    """Define dataset/training settings + paths as globals (needs REPO from setup())."""
    global DATASET_SOURCE, N_TRAIN, N_TEST, MALICIOUS_RATIO
    global KAGGLE_DATASET, CICIDS_MAX_PER_CLASS
    global MAX_LENGTH, BATCH_SIZE, NUM_EPOCHS, TRAIN_BERT
    global DATA_DIR, MODEL_DIR, TRAIN_JSON, TEST_JSON

    # ─── the one knob: where dataset/train_data2.json comes from ──────────────
    DATASET_SOURCE  = "synthetic"      # "synthetic" | "cicids2017"
    # synthetic sizes
    N_TRAIN         = 20000
    N_TEST          = 4000
    MALICIOUS_RATIO = 0.35
    # CIC-IDS2017 (only used when DATASET_SOURCE == "cicids2017")
    KAGGLE_DATASET       = "chethuhn/network-intrusion-dataset"
    CICIDS_MAX_PER_CLASS = 20000
    # detector training
    MAX_LENGTH = 512
    BATCH_SIZE = 16
    NUM_EPOCHS = 2
    TRAIN_BERT = False                 # heavy, and not used by the PPO reward
    # paths (match the repo's ../dataset and ../model conventions)
    DATA_DIR   = os.path.join(REPO, "dataset")
    MODEL_DIR  = os.path.join(REPO, "model")
    TRAIN_JSON = os.path.join(DATA_DIR, "train_data2.json")
    TEST_JSON  = os.path.join(DATA_DIR, "test2.json")
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)

def prepare():
    """setup() + config() — call once at the top of any stage to make it self-contained."""
    setup(); config()

config()
print("dataset source:", DATASET_SOURCE, "| model dir:", MODEL_DIR)

## Install — run once, then no restarts

One pinned stack for all four stages (see the comments for why each version). ~5 min the first time (it downgrades torch to 2.3.1 and pulls unsloth + CUDA deps). Run this **before Stage 2**.

In [ ]:
# ── One-time install: the full pinned stack for ALL stages (Colab Python 3.11) ──
# Shared core comes from the repo's requirements.txt (single source of truth, with the
# reason for every pin). torch + xformers are installed separately because they need
# special handling (see the requirements.txt header):
#   torch 2.3.1 WITH deps -> pulls cuDNN 8 + triton 2.3.1 (else "libcudnn.so.8 not found")
#   xformers 0.0.26.post1 --no-deps -> matches torch 2.3.1, ignores its hard torch==2.3.0 pin
# Runs after Setup (so REPO is set) and before Stage 2's first torch import -> no restarts.
!pip -q install -r {REPO}/requirements.txt
!pip -q install "torch==2.3.1" "torchvision==0.18.1"
!pip -q install --no-deps "xformers==0.0.26.post1"
print(">> stack ready: torch 2.3.1 | trl 0.8.6 | transformers 4.44.2 | accelerate 0.33.0 | bnb 0.43.1 | xformers 0.0.26.post1")

## Stage 1 — Dataset

Produces `dataset/train_data2.json` (+ `test2.json`) — the single file every downstream stage reads.

In [ ]:
import shutil
prepare()   # re-establish repo + config, so this stage runs standalone

if DATASET_SOURCE == "synthetic":
    !python gen_synthetic_data.py --n {N_TRAIN} --test-n {N_TEST} \
        --malicious-ratio {MALICIOUS_RATIO} --out "{TRAIN_JSON}" --test-out "{TEST_JSON}"

elif DATASET_SOURCE == "cicids2017":
    # needs a Kaggle token at ~/.kaggle/kaggle.json (Kaggle ▸ Settings ▸ API ▸ Create New Token)
    !pip -q install kaggle
    !apt-get -qq install -y tshark >/dev/null
    !kaggle datasets download -d {KAGGLE_DATASET} -p /content/cicids --unzip
    # PCAPs carry the HTTP text; TrafficLabelling CSVs carry the labels (join on the 5-tuple).
    !python build_train_data.py \
        --pcap-dir /content/cicids/PCAPs \
        --csv-dir  /content/cicids/TrafficLabelling \
        --max-per-class {CICIDS_MAX_PER_CLASS} \
        --out "{TRAIN_JSON}"
    shutil.copy(TRAIN_JSON, TEST_JSON)   # PPO reads a held-out file; reuse the extract

else:
    raise ValueError("DATASET_SOURCE must be 'synthetic' or 'cicids2017'")

In [ ]:
import json, collections
recs = json.load(open(TRAIN_JSON, encoding="utf-8"))
print(len(recs), "records", dict(collections.Counter(r["Label"] for r in recs)))
r = recs[0]
print("\n" + r["Request Line"])
for k, v in list(r["Request Headers"].items())[:4]:
    print(f"  {k}: {v}")
print("  ->", r["Label"], "| source:", r["Source"])

## Stage 2 — Detector training

Trains the token-level (TextCNN / CNN-LSTM / DNN) and image detectors, then writes the `model_configs` pickles the PPO stage attacks. Runs on CPU or GPU.

In [ ]:
# token-level detectors (TextCNN / CNN-LSTM / DNN), sharing the BERT tokenizer's vocab
prepare()   # re-establish repo + config (must run before the DL imports below)
import os, torch
import DL.training as _T
from transformers import TrainingArguments, AutoTokenizer
from DL.data_processing import load_data, prepare_dataset
from DL.models import TextCNNClassifier, CNNLSTMClassifier, DNNClassifier
from DL.training import train_custom_model, train_transformer_model

_T.MODEL_PATH = MODEL_DIR          # repo hardcodes ./models/; redirect saves under model/
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

TOKENIZER_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
data = load_data(TRAIN_JSON)
train_ds, val_ds, test_ds = prepare_dataset(data, tokenizer, MAX_LENGTH)

vocab_size, embed_size, num_classes = len(tokenizer.vocab), 128, 2
os.makedirs(os.path.join(MODEL_DIR, "custom_models"), exist_ok=True)
args = TrainingArguments(output_dir=os.path.join(MODEL_DIR, "custom_models"),
                         per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
                         learning_rate=2e-5, num_train_epochs=NUM_EPOCHS, report_to="none")

for name, model in {
        "textcnn":  TextCNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH),
        "cnn_lstm": CNNLSTMClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH),
        "dnn":      DNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)}.items():
    print("training", name)
    train_custom_model(model, name, train_ds, val_ds, args)

if TRAIN_BERT:
    bargs = TrainingArguments(output_dir=os.path.join(MODEL_DIR, "bert"),
                              evaluation_strategy="epoch", learning_rate=2e-5,
                              per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
                              num_train_epochs=NUM_EPOCHS, weight_decay=0.01, save_strategy="epoch",
                              load_best_model_at_end=True, report_to="none")
    train_transformer_model("bert", TOKENIZER_NAME, train_ds, val_ds, bargs)

In [ ]:
# image-based detectors: each request rendered as a 28x28 byte image (ord(c) % 128)
import numpy as np, torch, os
from torch.utils.data import DataLoader, TensorDataset
from DL.data_processing import json_to_string
from DL.image_models import ImageCNN, ImageMLP

IMG = (28, 28)
def to_image(it):
    text = it["Request Line"] + "\n" + json_to_string(it["Request Headers"]) + "\n\n" + it["Request Body"]
    v = [ord(c) % 128 for c in text][:IMG[0] * IMG[1]]
    v += [0] * (IMG[0] * IMG[1] - len(v))
    return np.array(v, dtype=np.float32).reshape(IMG)

X = torch.tensor(np.stack([to_image(r) for r in data]))
y = torch.tensor([1 if r["Label"] == "Malicious" else 0 for r in data], dtype=torch.long)
k = int(len(X) * 0.9)
loader = DataLoader(TensorDataset(X[:k], y[:k]), batch_size=64, shuffle=True)

for name, m in {"imagecnn": ImageCNN(), "imagemlp": ImageMLP()}.items():
    m.to(device); opt = torch.optim.Adam(m.parameters(), 1e-3); lf = torch.nn.CrossEntropyLoss()
    for _ in range(int(NUM_EPOCHS)):
        m.train()
        for xb, yb in loader:
            loss = lf(m(xb.to(device)), yb.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad():
        acc = (m(X[k:].to(device)).argmax(1).cpu() == y[k:]).float().mean().item()
    path = os.path.join(MODEL_DIR, "custom_models", name + ".bin")
    torch.save(m.state_dict(), path)
    print(f"{name}: val acc {acc:.3f} -> {path}")

In [ ]:
import pickle, os
from transformers import AutoTokenizer
from DL.models import TextCNNClassifier, CNNLSTMClassifier, DNNClassifier
from DL.image_models import ImageCNN, ImageMLP

cm = os.path.join(MODEL_DIR, "custom_models")

def cfg(name, cls):
    return {"type": "custom", "name": name, "path": os.path.join(cm, name + ".bin"), "class": cls}

text_configs = [
    cfg("textcnn",  TextCNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)),
    cfg("cnn_lstm", CNNLSTMClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)),
    cfg("dnn",      DNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)),
]
image_configs = [cfg("imagecnn", ImageCNN()), cfg("imagemlp", ImageMLP())]

pickle.dump(text_configs,  open(os.path.join(MODEL_DIR, "model_configs.pkl"), "wb"))
pickle.dump(image_configs, open(os.path.join(MODEL_DIR, "imgae_model_configs.pkl"), "wb"))

# PPO's Text reward re-tokenises responses with this — save it even when BERT is skipped.
AutoTokenizer.from_pretrained(TOKENIZER_NAME).save_pretrained(os.path.join(MODEL_DIR, "bert"))
print("wrote model_configs.pkl, imgae_model_configs.pkl, model/bert/ tokenizer")

## Stage 3 — LLM finetuning (Unsloth, GPU)

LoRA SFT of Llama-3-8b to generate benign/malicious traffic in the dataset's format. Fits a T4. Uses the stack from the **Install** cell (unsloth + torch 2.3.1 + xformers); saves the adapter to `model/llama_lora`.

In [ ]:
import os, json, torch
prepare()   # re-establish repo + config, so this stage runs standalone
assert torch.cuda.is_available(), "Stage 3 needs a GPU runtime"

from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

MAX_SEQ = 2048
model, tok = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit", max_seq_length=MAX_SEQ,
    dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407)

def json_to_string(d, indent=0):
    out, pad = [], " " * indent
    if isinstance(d, dict):
        for k, v in d.items():
            if isinstance(v, (dict, list)):
                out.append(f"{pad}{k}:"); out.append(json_to_string(v, indent + 1))
            else:
                out.append(f"{pad}{k}: {v}")
    elif isinstance(d, list):
        for it in d:
            out.append(json_to_string(it, indent))
    else:
        out.append(f"{pad}{d}")
    return "\n".join(out)

alpaca = ("Below is an instruction that describes a task, paired with an input that provides "
          "further context. Write a response that appropriately completes the request.\n\n"
          "### Instruction:\n{}\n\n### Input:\n{}\n\n### Response:\n{}")
EOS = tok.eos_token
data = json.load(open(TRAIN_JSON, encoding="utf-8"))
def body(it): return it["Request Line"] + "\n" + json_to_string(it["Request Headers"]) + "\n\n" + it["Request Body"]
texts = [alpaca.format(
            "Follow these tips to generate malicious http traffic" if it["Label"] == "Malicious"
            else "Follow these tips to generate benign http traffic",
            it["Request Line"], body(it)) + EOS
         for it in data]
ds = Dataset.from_dict({"text": texts}).shuffle(seed=42)

trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=ds,
    dataset_text_field="text", max_seq_length=MAX_SEQ, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        warmup_steps=5, max_steps=60, learning_rate=2e-4,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=5, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407,
        output_dir=os.path.join(MODEL_DIR, "llama_outputs"), report_to="none"))
trainer.train()
model.save_pretrained(os.path.join(MODEL_DIR, "llama_lora"))
tok.save_pretrained(os.path.join(MODEL_DIR, "llama_lora"))
print("saved LoRA ->", os.path.join(MODEL_DIR, "llama_lora"))

## Stage 4 — PPO adversarial generation (GPU)

PPO tunes a generator (`EleutherAI/pythia-160m`) to flip the frozen detectors' predictions. Reward = detector score for the *opposite* label. `FEATURE_TYPE` picks which detector family to attack. Uses the **same stack as Stage 3** — no restart needed.

In [ ]:
import os, sys, pickle, torch
import torch.nn.functional as F

prepare()   # re-establish repo + config, so this stage runs standalone
RL_DIR = os.path.join(REPO, "RL-Adv")
sys.path.insert(0, RL_DIR)
os.chdir(RL_DIR)   # so the repo's ../model and ../dataset paths resolve

from config import (create_ppo_config, features_dict, device, generation_kwargs,
                    columns_to_log, output_min_length, output_max_length,
                    max_length, query_max_length)
from data_utils import load_http_dataset, create_dataloader, text2image
from model_utils import setup_models, prepare_query_tensors, evaluate_responses
from utils import set_seed, save_results, mkdir
from trl import PPOTrainer
from trl.core import LengthSampler

FEATURE_TYPE  = "Text"    # "Text" | "Image" — which detector family to attack
MAX_PPO_STEPS = 40
SAMPLE_SIZE   = 2000

set_seed(42)
# note: named ppo_config (not `config`) so it doesn't clobber the config() function prepare() uses
ppo_config = create_ppo_config()
ppo_config.is_peft_model = False   # PPO a plain fp16 policy, not a LoRA adapter
ppo_config.log_with = None         # skip wandb

dataset    = load_http_dataset(file_path="../dataset/test2.json", sample_size=SAMPLE_SIZE)
dataloader = create_dataloader(dataset, batch_size=ppo_config.batch_size)

ppo_model, ref_model, tokenizer = setup_models(ppo_config.model_name, device, load_in_4bit=False)
generation_kwargs["pad_token_id"] = tokenizer.eos_token_id
generation_kwargs["top_k"] = 0
ppo_trainer = PPOTrainer(ppo_config, ppo_model, ref_model, tokenizer, dataset=dataset)

model_configs = pickle.load(open(features_dict[FEATURE_TYPE], "rb"))  # skips the input() prompt
out_sampler   = LengthSampler(output_min_length, output_max_length)

test_tokenizer = None
if FEATURE_TYPE == "Text":
    from transformers import AutoTokenizer
    test_tokenizer = AutoTokenizer.from_pretrained("../model/bert/")

all_data = []
for step, batch in enumerate(dataloader):
    if step >= MAX_PPO_STEPS:
        break
    query_tensors, origin_label, requirement_label, _ = prepare_query_tensors(
        batch, tokenizer, device, query_max_length)
    generation_kwargs["max_new_tokens"] = out_sampler()
    resp = ppo_trainer.generate(query_tensors, **generation_kwargs)
    response_tensors = [r.squeeze()[:max_length] for r in resp]
    batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]

    if FEATURE_TYPE == "Text":
        texts  = [r.split("\n", 1)[-1][:max_length] for r in batch["response"]]
        tt     = [torch.tensor(test_tokenizer(t)["input_ids"]).to(device) for t in texts]
        padded = [F.pad(t, (0, max_length - t.size(0))) if t.size(0) < max_length else t[:max_length] for t in tt]
        feats  = torch.stack(padded)
    else:
        feats  = torch.stack(text2image(batch["response"])).to(device)

    rewards, pred = evaluate_responses(batch, FEATURE_TYPE, model_configs, feats, device, requirement_label)
    stats = ppo_trainer.step(query_tensors, response_tensors, list(rewards))
    ppo_trainer.log_stats(stats, batch, rewards, columns_to_log=columns_to_log)

    hit = (pred.cpu() == torch.tensor(requirement_label)).float().mean().item()
    print(f"step {step:02d}  reward {rewards.mean():.3f}  target-hit {hit:.2f}")

    for i in range(len(batch["instruction"])):
        all_data.append({"Request Line": batch["input"][i], "Label": origin_label[i],
                         "Origin Output": batch["output"][i], "Request Body": "",
                         "Request Headers": batch["response"][i]})

save_results(all_data, FEATURE_TYPE, 0)
save_path = os.path.join("../model/ppo_model", FEATURE_TYPE)
mkdir(save_path)
ppo_model.save_pretrained(save_path); tokenizer.save_pretrained(save_path)
os.chdir(REPO)
print("saved PPO policy ->", os.path.join(MODEL_DIR, "ppo_model", FEATURE_TYPE))

In [ ]:
import glob, json, os

files = sorted(glob.glob(os.path.join(REPO, "dataset", "PPO_data", "**", "*.json"), recursive=True))
if files:
    print("latest:", files[-1])
    print(json.dumps(json.load(open(files[-1]))[:2], indent=2)[:1500])
else:
    print("no PPO_data yet — run Stage 4 first")

In [ ]:
# ── Stage 4b (faithful): PPO-tune the Stage-3 Llama instead of pythia-160m ──
# Needs model/llama_lora from Stage 3 on disk (same session), and the Stage 4 stack.
# Checkpoints to model/ppo_llama_ckpt and auto-resumes if re-run. Tiny "prove it runs" config.
import os, sys, importlib
RL_DIR = os.path.join(REPO, "RL-Adv")
sys.path.insert(0, RL_DIR)
os.chdir(RL_DIR)
import ppo_llama; importlib.reload(ppo_llama)
asr = ppo_llama.run(steps=15, sample_size=256, feature_type="Text", batch_size=2)
os.chdir(REPO)
print("final ASR:", asr)